# Predicción de Gasto de Gas y Comparativa Sistema Híbrido
Este notebook toma el histórico de consumo de gas para entrenar un modelo de Machine Learning (`LinearRegression`) y estimar tu consumo para los próximos 5 y 15 años. Además, compara los costes proyectados (Peor Escenario Posible debido a impuestos de CO₂) contra la instalación de un sistema híbrido.

In [ ]:
%pip install pandas plotly scikit-learn openpyxl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

## 1. Cargar datos de consumo histórico

In [ ]:
url_excel = 'data/export_zaehlerstandshistorie_891288_20261003.xlsx'
df = pd.read_excel(url_excel)
df = df.rename(columns={'Unnamed: 0': 'Fecha', 'Unnamed: 4': 'Zaehlerstand', 'Unnamed: 5': 'Verbrauch'})
df = df[['Fecha', 'Verbrauch']]

df['Fecha'] = pd.to_datetime(df['Fecha'], format='%d.%m.%Y', errors='coerce')
df['Verbrauch'] = pd.to_numeric(df['Verbrauch'], errors='coerce')
df = df.dropna()
df['Anio'] = df['Fecha'].dt.year

# Tomamos los últimos 10 años
df = df[df['Anio'] >= (df['Anio'].max() - 10)]
df = df.sort_values('Anio').reset_index(drop=True)

df_anual = df.groupby('Anio')['Verbrauch'].sum().reset_index()
df_anual.tail()

## 2. Entrenamiento del Modelo de IA
Usamos `LinearRegression` solo con el Año como característica.

In [ ]:
X = df_anual[['Anio']]
y = df_anual['Verbrauch']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

mejor_modelo = LinearRegression()
mejor_modelo.fit(X_scaled, y)

print('¡Modelo entrenado con éxito!')

## 3. Predicción a 5 Años (2027 - 2031) y Análisis del Peor Escenario
Proyectamos un escenario donde los precios del gas escalan hasta 22 céntimos de euro por kWh para el final de la década.

In [ ]:
anios_futuros_5 = pd.DataFrame({'Anio': [2027, 2028, 2029, 2030, 2031]})
X_futuro_5_scaled = scaler.transform(anios_futuros_5[['Anio']])

consumo_predicho_5 = mejor_modelo.predict(X_futuro_5_scaled)
anios_futuros_5['Consumo_m3_Estimado'] = consumo_predicho_5

# Precios del peor escenario (EUR por m3 = céntimos por kWh * 10)
precios_peor_caso_kwh = [0.14, 0.16, 0.18, 0.20, 0.22]
precios_peor_caso_m3 = [p * 10 for p in precios_peor_caso_kwh]
anios_futuros_5['Precio_Est_m3_EUR'] = precios_peor_caso_m3

anios_futuros_5['Costo_Total_EUR'] = anios_futuros_5['Consumo_m3_Estimado'] * anios_futuros_5['Precio_Est_m3_EUR']
anios_futuros_5

## 4. Comparativa de Inversión (15 Años): Gas vs Sistema Híbrido
Extendemos la predicción a 15 años para observar el punto donde cruzan rentabilidades.

In [ ]:
anios_futuros = pd.DataFrame({'Anio': np.arange(2027, 2042)})
X_futuro_scaled = scaler.transform(anios_futuros[['Anio']])
anios_futuros['Consumo_m3_Estimado'] = mejor_modelo.predict(X_futuro_scaled)

precios_gas = [1.40, 1.60, 1.80, 2.00, 2.20, 2.30, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40, 2.40]
anios_futuros['Precio_Est_Gas_m3'] = precios_gas

# Opción 1: Solo Gas
costo_instalacion_gas = 40000
anios_futuros['Gasto_Factura_Gas'] = anios_futuros['Consumo_m3_Estimado'] * anios_futuros['Precio_Est_Gas_m3']
anios_futuros['Acumulado_Solo_Gas'] = costo_instalacion_gas + anios_futuros['Gasto_Factura_Gas'].cumsum()

# Opción 2: Sistema Híbrido
anios_futuros['Demanda_Calor_kWh'] = anios_futuros['Consumo_m3_Estimado'] * 10
anios_futuros['Elec_WP_kWh'] = (anios_futuros['Demanda_Calor_kWh'] * 0.8) / 3.5
anios_futuros['Gas_Hibrido_m3'] = anios_futuros['Consumo_m3_Estimado'] * 0.2

precio_elec = 0.30
anios_futuros['Gasto_Factura_Hibrido'] = (anios_futuros['Elec_WP_kWh'] * precio_elec) + (anios_futuros['Gas_Hibrido_m3'] * anios_futuros['Precio_Est_Gas_m3'])

costo_instalacion_hibrido_sin_sub = 92500
descuento_hibrido = 0.30
costo_instalacion_hibrido_con_sub = costo_instalacion_hibrido_sin_sub * (1 - descuento_hibrido)

anios_futuros['Acumulado_Hibrido_Sin_Sub'] = costo_instalacion_hibrido_sin_sub + anios_futuros['Gasto_Factura_Hibrido'].cumsum()
anios_futuros['Acumulado_Hibrido_Con_Sub'] = costo_instalacion_hibrido_con_sub + anios_futuros['Gasto_Factura_Hibrido'].cumsum()

# Gráfica
fig = go.Figure()
fig.add_trace(go.Scatter(x=anios_futuros['Anio'], y=anios_futuros['Acumulado_Solo_Gas'], mode='lines+markers', name='Gas (Instalación Nueva + Consumo)', line=dict(color='red')))
fig.add_trace(go.Scatter(x=anios_futuros['Anio'], y=anios_futuros['Acumulado_Hibrido_Sin_Sub'], mode='lines+markers', name='Híbrido (Sin Subvención)', line=dict(color='orange', dash='dash')))
fig.add_trace(go.Scatter(x=anios_futuros['Anio'], y=anios_futuros['Acumulado_Hibrido_Con_Sub'], mode='lines+markers', name='Híbrido (Con Subvención)', line=dict(color='green')))
fig.update_layout(title='Evolución del Coste Acumulado (15 Años)', xaxis_title='Año', yaxis_title='Coste Acumulado (€)')
fig.show()
